In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt
from tqdm import tqdm
import shutil

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [2]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

Latest run date: 2025-03-10 20:47:16.070018


#### Functions

In [3]:
def map_ltv_range_to_lgd_bin(flt_ltv, dict_bins_ltv):
    for flt_threshold, flt_val in dict_bins_ltv.items():
        if flt_ltv <= flt_threshold:
            return flt_val
    # else
    return np.max(list(dict_bins_ltv.values()))

In [4]:
def get_lgd_bk_nobk(int_bk, flt_lgd_bk, flt_lgd_nobk):
    # if bk
    if int_bk == 1:
        return flt_lgd_bk
    else:
        return flt_lgd_nobk

#### Constants

In [5]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

Project: 20250307-funded-trends
Task: 06_get_gen13_predictions


#### Make output dir

In [6]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [7]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/04_join_targets/{str_filename}'
df = pd.read_parquet(str_uri)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,fltNetChgOff_2,fltNetChgOff_3,fltNetChgOff_6,fltNetChgOff_12,fltNetChgOff_24,co_at_60,co_at_90,co_at_180,co_at_360,co_at_720
0,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0.0,0.0,0.0,0.0,7827.16,0.0,0.0,0.0,0.0,0.315976
1,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
2,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
3,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
4,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100027,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100028,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100029,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000


#### Get Gen 13 Predictions

In [8]:
# import parser
str_filename = 'cls_parser.pkl'
str_local_path = f'./{str_filename}'
cls_parser = pickle.load(open(str_local_path, 'rb'))

# preprocess
cls_model_preprocessing = cls_parser.cls_model_preprocessing
df = cls_model_preprocessing.transform(df)
# show
df

Masking negative values to NaN...


100%|██████████| 2115/2115 [00:03<00:00, 703.53it/s]


Capping income...
Replacing zeros...


100%|██████████| 3/3 [00:00<00:00, 1028.77it/s]


Engineering number of months...
Engineering number of months total...
Engineering weighted average...
Engineering tag for has auto...
Engineering tag for open auto indicator...
Engineering tag for closed auto indicator...
Engineering tag for open and closed auto indicator...
Engineering 3 month early delinquency...
Engineering 6 month early delinquency...
Engineering 3 month recent delinquency...
Engineering 6 month recent delinquency...
Engineering DTI...
Engineering franchise...
Engineering has a codebtor...
Engineering vehicle age...
Engineering PTI...
Engineering LTV...
Engineering BK...
Engineering perfect payment history tag for most recent auto...
Engineering perfect payment history tag for open auto...
Engineering perfect payment history tag for closed auto...
Engineering interactions...
Imputing values...


100%|██████████| 1123/1123 [00:01<00:00, 1019.88it/s]


Binning values for scorecard...


100%|██████████| 1119/1119 [00:04<00:00, 276.95it/s]


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,addrchangecount06month__ln_binned,addrchangecount12month__ln_binned,addrchangecount24month__ln_binned,addrchangecount60month__ln_binned,addrlastmovetaxratiodiff__ln_binned,addrlastmoveecontrajectory__ln_binned,addrlastmoveecontrajectoryindex__ln_binned,phoneinputproblems__ln_binned,phoneinputsubjectcount__ln_binned,alertregulatorycondition__ln_binned
0,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0.0,0.048333,0.108514,0.032394,0.0,-0.097823,-0.091170,0.032249,0.028913,0.216812
1,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.0,0.048333,0.108514,0.225637,0.0,0.211757,0.172319,-0.039639,-0.038041,0.216812
2,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0.0,-0.211617,-0.158847,-0.289687,0.0,-0.097823,-0.134102,-0.039639,-0.038041,0.216812
3,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0.0,0.048333,0.108514,0.225637,0.0,0.211757,0.172319,0.032249,0.028913,-0.107256
4,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0.0,-0.211617,-0.266322,-0.257352,0.0,-0.097823,-0.134102,0.032249,0.028913,-0.107256
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0.0,0.048333,0.108514,0.225637,0.0,0.211757,0.172319,-0.039639,-0.038041,-0.107256
100027,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0.0,0.048333,0.108514,0.225637,0.0,0.211757,0.172319,-0.039639,-0.038041,-0.107256
100028,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0.0,0.048333,0.108514,0.225637,0.0,0.211757,0.172319,0.032249,0.028913,-0.107256
100029,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0.0,0.048333,-0.158847,-0.257352,0.0,0.211757,-0.134102,0.032249,0.028913,0.216812


In [9]:
# predict - PD
cls_model_inference = cls_parser.cls_model_inference
# get intercept
flt_intercept = cls_model_inference.intercept_[0]
# get cols in model
list_cols_model = list(cls_model_inference.feature_names_in_)
# get the coef
list_coef = list(cls_model_inference.coef_[0])
# make dict
dict_coef = dict(zip(list_cols_model, list_coef))
# get contribution
list_str_contribution = []
for key, val in dict_coef.items():
    str_contribution = f'{key}_contribution'
    df[str_contribution] = df[key] * val 
    list_str_contribution.append(str_contribution)
# get the sum
df['sum'] = df[list_str_contribution].apply(sum, axis=1)
# get the log odds
df['log_odds'] = df['sum'] + flt_intercept
# get the pd
df['gen13_pd'] = np.exp(df['log_odds']) / (1 + np.exp(df['log_odds']))

In [10]:
# predict - LGD (BK)
dict_bins_ltv_bk = cls_parser.dict_bins_ltv_bk
df['LGD_bk'] = df['ENG-loan_to_value'].apply(
    lambda x: map_ltv_range_to_lgd_bin(
        flt_ltv=x,
        dict_bins_ltv=dict_bins_ltv_bk,
    ),
)

In [11]:
dict_bins_ltv_nobk = cls_parser.dict_bins_ltv_nobk
df['LGD_nobk'] = df['ENG-loan_to_value'].apply(
    lambda x: map_ltv_range_to_lgd_bin(
        flt_ltv=x,
        dict_bins_ltv=dict_bins_ltv_nobk,
    ),
)

In [12]:
# get lgd
df['gen13_lgd'] = df.apply(
    lambda x: get_lgd_bk_nobk(
        int_bk=x['ENG-bk'],
        flt_lgd_bk=x['LGD_bk'],
        flt_lgd_nobk=x['LGD_nobk'],
    ),
    axis=1,
)

#### Save to s3

In [13]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)